# Alexandria Entrypoints Smoke Notebook

This notebook is a manual smoke check for public ingest plus retrieve through the FastAPI boundary. It uses the configured Postgres database, real SQL repositories, real unit of work, the configured embedder, and the configured summarizer.

It does not start a long-running API server. `TestClient` exercises the same FastAPI routes in-process.

## Prerequisites

Expected setup:
- Docker and Task are available locally.
- The configured Postgres database is reachable through `.env` or the environment.
- Provider credentials for the configured embedder and summarizer are present in `.env` or the environment.
- The notebook is run from the repository root or from the `sandbox/` directory.

Unlike the deterministic automated smoke test, this notebook calls the real configured provider adapters. The inserted document source keys include a timestamp so the rows are easy to identify in a developer database.

## Start Infrastructure

This cell starts the local infrastructure profile through the project Taskfile and waits for Docker Compose health checks before the smoke path runs.

In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess


repo = Path.cwd()
if not (repo / "pyproject.toml").exists():
    repo = repo.parent

deploy = subprocess.run(
    ["task", "deploy", "--", "--wait"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

if deploy.stdout:
    print(deploy.stdout)
if deploy.stderr:
    print(deploy.stderr)


## Build The Public Client

This cell prepares configured storage and creates a public FastAPI client. Workflow routes construct request-scoped real `App` instances with configured storage, embedder, summarizer, repositories, search adapter, and unit of work. The leaf threshold is raised for this smoke check so these two documents do not intentionally trigger split work while public entrypoints are being exercised.

In [ ]:
from datetime import datetime, timezone

from fastapi.testclient import TestClient
from sqlalchemy import select

from domain.entity import Document, Node
from infrastructure.config import IngestSettings, get_settings
from infrastructure.persistence.db import Db
from presentation.api.app import api


get_settings.cache_clear()
base_settings = get_settings()
settings = base_settings.model_copy(update={"ingest": IngestSettings(max_leaf_docs=100)})
db = Db(settings.database)
db.create_all()
api_app = api()
api_app.state.settings = settings
client = TestClient(api_app)

stamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
source_alpha = f"notebook:entrypoints:alpha:{stamp}"
source_beta = f"notebook:entrypoints:beta:{stamp}"
marker = f"alexandria entrypoint smoke marker {stamp}"

print("app", settings.app.app_name, settings.app.app_version)
print("sources", source_alpha, source_beta)


## Confirm Public Health And Version

These calls prove the existing public utility endpoints still work through the same public API configuration used for workflow calls.

In [ ]:
health_response = client.get("/health")
version_response = client.get("/version")

assert health_response.status_code == 200
assert version_response.status_code == 200
print("health", health_response.json())
print("version", version_response.json())


## Ingest Through The Public API

These requests call `POST /ingest`. The route validates transport input, translates it to `DocIn`, and then the real app calls the configured summarizer and embedder before writing through the configured database.

In [ ]:
alpha_response = client.post(
    "/ingest",
    json={
        "name": f"Notebook Entry Alpha {stamp}",
        "body": f"Alpha document for {marker}. It should be retrievable through the public API.",
        "source_key": source_alpha,
    },
)
beta_response = client.post(
    "/ingest",
    json={
        "name": f"Notebook Entry Beta {stamp}",
        "body": f"Beta document for {marker}. It should be retrievable through the public API.",
        "source_key": source_beta,
    },
)

assert alpha_response.status_code == 200, alpha_response.text
assert beta_response.status_code == 200, beta_response.text
alpha_id = alpha_response.json()["id"]
beta_id = beta_response.json()["id"]

print("alpha", alpha_response.json())
print("beta", beta_response.json())


## Inspect Durable State After Ingest

This cell reads the configured database directly to prove the public request created durable document rows with provider-generated summaries and embeddings.

In [ ]:
with db.session() as session:
    docs = session.scalars(
        select(Document)
        .where(Document.source_key.in_((source_alpha, source_beta)))
        .order_by(Document.source_key.asc())
    ).all()
    leaf_ids = {doc.leaf_id for doc in docs}
    leaves = session.scalars(select(Node).where(Node.id.in_(leaf_ids))).all()

    print("documents", [(str(doc.id), doc.source_key, doc.name) for doc in docs])
    print("summaries", [(doc.source_key, doc.summary[:120]) for doc in docs])
    print("embedding_dimensions", [(doc.source_key, len(list(doc.embedding))) for doc in docs])
    print("leaves", [(str(leaf.id), leaf.kind, leaf.status, leaf.doc_count) for leaf in leaves])

    assert {str(doc.id) for doc in docs} == {alpha_id, beta_id}
    assert {doc.source_key for doc in docs} == {source_alpha, source_beta}
    assert all(doc.summary for doc in docs)
    assert all(len(list(doc.embedding)) > 0 for doc in docs)


## Retrieve Through The Public API

This request calls `GET /retrieve`. The route validates query input, calls `App.retrieve`, and serializes public document hit data.

In [ ]:
retrieve_response = client.get(
    "/retrieve",
    params={"query": marker, "limit": 25},
)
assert retrieve_response.status_code == 200, retrieve_response.text

payload = retrieve_response.json()
hits = payload["hits"]
hit_sources = [hit["source_key"] for hit in hits]
selected = [hit for hit in hits if hit["source_key"] in {source_alpha, source_beta}]

print("hit_sources", hit_sources)
print("selected", selected)

assert {source_alpha, source_beta}.issubset(set(hit_sources))
assert all("embedding" not in hit for hit in hits)
assert all("score" in hit and "distance" in hit and "bm25" in hit for hit in hits)
payload


### Expected Smoke Output

- `health` and `version` should return successful public responses.
- `alpha` and `beta` should print created document ids from `POST /ingest`.
- Durable state should show two inserted documents, non-empty summaries, stored embeddings, and leaf metadata.
- Retrieval should include both notebook source keys in the returned hit set.
- Public hit payloads should include document fields and scores, but not embeddings.

### Matching Pytest Smoke Command

Run this from the repository root:

```bash
uv run pytest tests/integration/test_entrypoint_flow.py -q
```

The pytest smoke remains deterministic and uses fake provider ports. This notebook is the provider-backed manual smoke path.

## Tear Down Infrastructure

Run this final cell when the smoke check is complete. It closes the in-process public client and stops the local infrastructure profile through the project Taskfile without deleting Docker volumes.

In [ ]:
from pathlib import Path
import subprocess


client.close()

repo = Path.cwd()
if not (repo / "pyproject.toml").exists():
    repo = repo.parent

teardown = subprocess.run(
    ["task", "teardown"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

if teardown.stdout:
    print(teardown.stdout)
if teardown.stderr:
    print(teardown.stderr)
